In [3]:
import torch
import clip
from PIL import Image

# 1. Check if the package is correctly imported
print(f"CLIP version: {clip.__version__ if hasattr(clip, '__version__') else 'Installed'}")

# 2. Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 3. Load a small model to test functionality
try:
    model, preprocess = clip.load("ViT-B/32", device=device)
    print("✅ Model loaded successfully!")
    
    # 4. List available models to verify full access
    print(f"Available models: {clip.available_models()}")
    
except Exception as e:
    print(f"❌ Error loading CLIP: {e}")


CLIP version: Installed
Using device: cuda
✅ Model loaded successfully!
Available models: ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']


In [4]:
%pip install -U click huggingface_hub

  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached huggingface_hub-1.9.0-py3-none-any.whl.metadata (14 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
Using cached click-8.3.2-py3-none-any.whl (108 kB)
Using cached huggingface_hub-1.9.0-py3-none-any.whl (637 kB)
Using cached typer-0.24.1-py3-none-any.whl (56 kB)
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.23.5
    Uninstalling huggingface-hub-0.23.5:
      Successfully uninstalled huggingface-hub-0.23.5
Note: you may need to restart the kernel to use updated packages.


In [5]:
# ============================================================
# CELL 2: IMPORTS, CONFIG, SEEDS, GPU
# ============================================================
import os, sys, warnings, io, random
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fft import fft2, fftshift
import pywt

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.base import BaseEstimator, ClassifierMixin
import xgboost as xgb
import lightgbm as lgb
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import clip
import timm
import albumentations as A

warnings.filterwarnings('ignore')

# ─── Dataset Paths (Renku) ──────────────────────────────
BASE_PATH = Path("DCU 2026 ML challenge - external 2")  # top-level unzipped folder
IMAGE_DIR = BASE_PATH / "images/images_final_sample"    # folder containing all images
TRAIN_CSV = BASE_PATH / "train.csv"                     # train CSV
TEST_CSV  = BASE_PATH / "test.csv"                      # test CSV

# Quick check to ensure paths are resolving correctly in Renku
print("--- PATH VERIFICATION ---")
print("BASE_PATH:", BASE_PATH)
print("IMAGE_DIR exists?", IMAGE_DIR.exists())
print("TRAIN_CSV exists?", TRAIN_CSV.exists())
print("TEST_CSV exists?", TEST_CSV.exists())
print("-------------------------\n")

# ─── Config ──────────────────────────────────────────────
SEED        = 42
CACHE_DIR   = Path("./feature_cache_v5")
MODEL_DIR   = Path("./saved_models_v5")
CACHE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

N_FOLDS          = 5
FORCE_FRESH      = False      # Set True to re-extract all features
CLIP_DIM         = 768
DINO_DIM         = 768
CNN_DIM          = 1280
FORENSIC_DIM     = 120        # ELA(30) + FFT(50) + Noise(40)

# ─── Reproducibility ─────────────────────────────────────
def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

--- PATH VERIFICATION ---
BASE_PATH: DCU 2026 ML challenge - external 2
IMAGE_DIR exists? True
TRAIN_CSV exists? True
TEST_CSV exists? True
-------------------------

Device: cuda
GPU: NVIDIA GeForce RTX 3090
VRAM: 25.4 GB


In [6]:
import logging
# Silence huggingface hub warnings
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# ... (the rest of your imports)

In [7]:
# ============================================================
# CELL 3: DATASET LOADING & DEEP EXPLORATION
# ============================================================

# ─── Load CSVs ───────────────────────────────────────────
df_train_raw = pd.read_csv(TRAIN_CSV)
df_test_raw = pd.read_csv(TEST_CSV)

sep = "=" * 60
print(sep)
print("DATASET OVERVIEW")
print(sep)

# ─── Train set ───────────────────────────────────────────
print("\n--- TRAIN SET ---")
print(f"Shape: {df_train_raw.shape}")
print(f"Columns: {df_train_raw.columns.tolist()}")
print("\nFirst 5 rows:")
print(df_train_raw.head())
print("\nClass distribution:")
print(df_train_raw['ground_truth'].value_counts())
n_real = (df_train_raw.ground_truth == 0).sum()
n_ai = (df_train_raw.ground_truth == 1).sum()
print(f"\nClass balance:")
print(f"  Real (0): {n_real} ({n_real/len(df_train_raw):.1%})")
print(f"  AI   (1): {n_ai} ({n_ai/len(df_train_raw):.1%})")

# ─── Test set ────────────────────────────────────────────
print("\n--- TEST SET ---")
print(f"Shape: {df_test_raw.shape}")
print(f"Columns: {df_test_raw.columns.tolist()}")
print("\nFirst 5 rows:")
print(df_test_raw.head())

# ─── Build filepaths ─────────────────────────────────────
df_train = df_train_raw.copy()
df_train['label'] = df_train['ground_truth'].astype(int)
df_train['filepath'] = df_train['image_id'].apply(lambda x: str(IMAGE_DIR / x))

df_test = df_test_raw.copy()
df_test['filepath'] = df_test['image_id'].apply(lambda x: str(IMAGE_DIR / x))

# ─── Verify paths exist ──────────────────────────────────
train_ok = sum(1 for p in df_train['filepath'].head(20) if Path(p).exists())
test_ok = sum(1 for p in df_test['filepath'].head(20) if Path(p).exists())
print("\n--- PATH VERIFICATION ---")
print(f"Train images found: {train_ok}/20 checked")
print(f"Test images found:  {test_ok}/20 checked")

if train_ok == 0:
    print(f"WARNING: No images found! Check IMAGE_DIR: {IMAGE_DIR}")

# ─── Image format analysis ───────────────────────────────
print("\n--- IMAGE FORMAT ANALYSIS ---")
train_ext = df_train['image_id'].apply(lambda x: Path(x).suffix.lower()).value_counts()
print("Train formats:")
print(train_ext)

# ─── Sample image sizes & properties ─────────────────────
print("\n--- SAMPLE IMAGE PROPERTIES ---")
sizes = []
modes = []
file_sizes = []
for p in df_train['filepath'].head(50):
    try:
        fp = Path(p)
        if fp.exists():
            file_sizes.append(fp.stat().st_size)
            with Image.open(p) as img:
                sizes.append(img.size)
                modes.append(img.mode)
    except Exception:
        pass

if sizes:
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    print(f"Width  range: {min(widths)} - {max(widths)} (median: {sorted(widths)[len(widths)//2]})")
    print(f"Height range: {min(heights)} - {max(heights)} (median: {sorted(heights)[len(heights)//2]})")
    unique_sizes = set(sizes)
    print(f"Unique sizes: {len(unique_sizes)}")
    if len(unique_sizes) <= 10:
        for s in sorted(unique_sizes):
            count = sizes.count(s)
            print(f"  {s[0]}x{s[1]}: {count} images")
    print(f"Color modes: {dict(pd.Series(modes).value_counts())}")
    
if file_sizes:
    print(f"File size range: {min(file_sizes)/1024:.1f} KB - {max(file_sizes)/1024:.1f} KB")
    print(f"Avg file size: {np.mean(file_sizes)/1024:.1f} KB")

# ─── Visualize sample images ─────────────────────────────
print("\n--- SAMPLE IMAGES ---")
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Sample Images: Top=Real (0), Bottom=AI (1)", fontsize=14)

real_samples = df_train[df_train['label'] == 0].head(5)
ai_samples = df_train[df_train['label'] == 1].head(5)

for i, (_, row) in enumerate(real_samples.iterrows()):
    try:
        with Image.open(row['filepath']) as img:
            axes[0, i].imshow(img)
            axes[0, i].set_title(f"Real", fontsize=9)
            axes[0, i].axis('off')
    except Exception:
        axes[0, i].text(0.5, 0.5, 'Failed', ha='center', va='center')
        axes[0, i].axis('off')

for i, (_, row) in enumerate(ai_samples.iterrows()):
    try:
        with Image.open(row['filepath']) as img:
            axes[1, i].imshow(img)
            axes[1, i].set_title(f"AI", fontsize=9)
            axes[1, i].axis('off')
    except Exception:
        axes[1, i].text(0.5, 0.5, 'Failed', ha='center', va='center')
        axes[1, i].axis('off')

plt.tight_layout()
plt.savefig("sample_images.png", dpi=100, bbox_inches="tight")
plt.show()

# ─── Summary ─────────────────────────────────────────────
print(f"\n{sep}")
print("SUMMARY")
print(sep)
print(f"Total train images: {len(df_train)}")
print(f"Total test images:  {len(df_test)}")
print(f"CV strategy:        {N_FOLDS}-fold stratified")
n_fold_train = int(len(df_train) * (1 - 1/N_FOLDS))
n_fold_val = len(df_train) - n_fold_train
print(f"Per fold:           {n_fold_train} train / {n_fold_val} val")
print(f"Submission format:  image_id, ground_truth")

DATASET OVERVIEW

--- TRAIN SET ---
Shape: (4800, 2)
Columns: ['image_id', 'ground_truth']

First 5 rows:
                                   image_id  ground_truth
0  e7ea0752-f24a-4dac-926c-371fda631a0f.jpg             1
1  cdc7b8e5-1609-4f88-a20d-0dd321f8f489.jpg             0
2  c954e18f-8ab3-4bff-91d7-a0d7ff000e51.jpg             0
3  1c5fb5d5-75e0-431f-a03e-e50026f23fe3.jpg             0
4  fd0dd104-7300-4da3-a46f-6980531ead33.jpg             1

Class distribution:
ground_truth
0    2485
1    2315
Name: count, dtype: int64

Class balance:
  Real (0): 2485 (51.8%)
  AI   (1): 2315 (48.2%)

--- TEST SET ---
Shape: (2058, 1)
Columns: ['image_id']

First 5 rows:
                                   image_id
0  3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg
1  2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg
2  01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg
3  ac784910-b461-498d-b3a8-50b1e4116b11.jpg
4  6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg

--- PATH VERIFICATION ---
Train images found: 20/20 checked
Tes

In [39]:
# ============================================================
# CELL 4: IMAGE LOADING & AUGMENTATION UTILITIES
# ============================================================

# ── PIL-first image loading (Kaggle compatible) ──────────────
def load_image_pil(path):
    """Load image as PIL RGB. Falls back to cv2."""
    try:
        return Image.open(str(path)).convert('RGB')
    except Exception:
        try:
            img = cv2.imread(str(path))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                return Image.fromarray(img)
        except Exception:
            pass
    return None

def load_image_np(path, size=None):
    """Load image as numpy array (RGB, float64, 0-255)."""
    img = load_image_pil(path)
    if img is None:
        return None
    if size:
        img = img.resize(size, Image.LANCZOS)
    return np.array(img, dtype=np.float64)

# ── CLIP transforms ──────────────────────────────────────────
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

# ── DINOv2 transforms ───────────────────────────────────────
DINO_MEAN = [0.485, 0.456, 0.406]
DINO_STD  = [0.229, 0.224, 0.225]

# ── Training augmentation (shared across fine-tuning) ────────
# KEY CHANGE: Added ImageCompression for forensic robustness
def get_train_transform_albu(mean, std, size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussNoise(var_limit=(5, 30), p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
        A.Normalize(mean=mean, std=std),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    ])

def get_val_transform_albu(mean, std, size=224):
    return A.Compose([
        A.Resize(height=256, width=256),
        A.CenterCrop(height=size, width=size),
        A.Normalize(mean=mean, std=std),
    ])

def get_tta_transform_albu(mean, std, size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.9, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=mean, std=std),
    ])

# ── Dataset class for albumentations ─────────────────────────
class AlbuDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = load_image_pil(self.paths[idx])
        if img is None:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        img_np = np.array(img)  # albumentations expects numpy
        augmented = self.transform(image=img_np)
        img_tensor = torch.from_numpy(augmented['image'].transpose(2, 0, 1)).float()

        lbl = self.labels[idx] if self.labels is not None else -1
        return img_tensor, torch.tensor(lbl, dtype=torch.float32)

# ── Mixup utility ────────────────────────────────────────────
def mixup_data(x, y, alpha=0.3):
    """Apply mixup augmentation on a batch."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute mixup loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("Augmentation utilities ready.")
print("  Key addition: ImageCompression(70-100) for forensic robustness")
print("  Key addition: Mixup (alpha=0.3) for regularization")

Augmentation utilities ready.
  Key addition: ImageCompression(70-100) for forensic robustness
  Key addition: Mixup (alpha=0.3) for regularization


In [9]:
# ============================================================
# CELL 5: FEATURE EXTRACTION — CLIP ViT-L/14 (768-dim)
# ============================================================
# Extract frozen CLIP embeddings for classical models.
# These are ALSO used as the starting point for fine-tuning.

set_seeds()

def extract_clip_features(image_paths, batch_size=64):
    """Extract L2-normalized CLIP ViT-L/14 embeddings."""
    # Load model to the RTX 3090
    model, preprocess = clip.load('ViT-L/14', device=DEVICE)
    
    # OPTIMIZATION: On RTX 3090, we can use .half() for faster extraction
    # if you prefer maximum precision, you can keep it .float()
    model = model.half() if DEVICE == "cuda" else model.float()
    model.eval()

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CLIP Extraction'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            # Ensure load_image_pil was defined in a previous cell
            img = load_image_pil(p)
            if img is not None:
                batch_imgs.append(preprocess(img))
            else:
                # Placeholder for missing images to maintain index alignment
                batch_imgs.append(torch.zeros(3, 224, 224))
        
        # Move batch to GPU
        batch = torch.stack(batch_imgs).to(DEVICE)
        if DEVICE == "cuda":
            batch = batch.half()

        with torch.no_grad():
            feats = model.encode_image(batch).float()
            
        # L2 normalize the features
        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    # FREE VRAM immediately - very important in shared computing environments
    del model
    torch.cuda.empty_cache()

    return np.vstack(feats_all).astype(np.float32)

# ── Extract & Cache ──────────────────────────────────────────
# Ensure df_train and df_test are already loaded from your path-fix cell
train_paths = df_train['filepath'].tolist()
test_paths  = df_test['filepath'].tolist()

if FORCE_FRESH or not (CACHE_DIR/'clip_train.npy').exists():
    print(f"Extracting CLIP features on {torch.cuda.get_device_name(0)}...")
    clip_train = extract_clip_features(train_paths)
    clip_test  = extract_clip_features(test_paths)
    
    # Save to your Renku project folder
    np.save(CACHE_DIR/'clip_train.npy', clip_train)
    np.save(CACHE_DIR/'clip_test.npy', clip_test)
    print("Features extracted and saved to cache.")
else:
    print("Loading CLIP features from cache...")
    clip_train = np.load(CACHE_DIR/'clip_train.npy')
    clip_test  = np.load(CACHE_DIR/'clip_test.npy')

print(f"CLIP train: {clip_train.shape}  test: {clip_test.shape}")

Loading CLIP features from cache...
CLIP train: (4800, 768)  test: (2058, 768)


In [1]:
%pip install -q huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [11]:
# ============================================================
# CELL 6: FEATURE EXTRACTION — DINOv2 ViT-B/14 (768-dim)
# ============================================================
import torch
import torchvision.transforms as T
from tqdm.auto import tqdm
import numpy as np

# Ensure Hugging Face hub is ready for timm
try:
    import huggingface_hub
except ImportError:
    %pip install -q huggingface_hub

def extract_dino_features(image_paths, batch_size=64): 
    """
    Extract L2-normalized DINOv2 ViT-B/14 embeddings.
    Optimized for RTX 3090 with FP16 and large resolution.
    """
    print(f"Loading DINOv2 model to {DEVICE}...")
    # vit_base_patch14_dinov2.lvd142m produces 768-dim embeddings
    model = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                              pretrained=True, num_classes=0)
    model = model.to(DEVICE).eval()

    # DINOv2 is optimized for 518x518 (must be multiple of patch size 14)
    tfm = T.Compose([
        T.Resize(518, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(518),
        T.ToTensor(),
        T.Normalize(DINO_MEAN, DINO_STD),
    ])

    feats_all = []
    
    # Process in batches
    for i in tqdm(range(0, len(image_paths), batch_size), desc='DINOv2 Extraction'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            try:
                img = load_image_pil(p)
                if img is not None:
                    batch_imgs.append(tfm(img.convert('RGB')))
                else:
                    batch_imgs.append(torch.zeros(3, 518, 518))
            except Exception:
                batch_imgs.append(torch.zeros(3, 518, 518))
        
        batch = torch.stack(batch_imgs).to(DEVICE)
        
        # Inference using AMP (Automatic Mixed Precision) for RTX 3090 speed
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                feats = model(batch)
            
        # L2 normalize for cosine similarity / cleaner training
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-6)
        feats_all.append(feats.float().cpu().numpy())

    # Strict VRAM management
    del model
    torch.cuda.empty_cache()
    
    return np.vstack(feats_all).astype(np.float32)

# ── Execution ───────────────────────────────────────────────
if FORCE_FRESH or not (CACHE_DIR/'dino_train.npy').exists():
    print(f"Starting DINOv2 extraction on {torch.cuda.get_device_name(0)}...")
    dino_train = extract_dino_features(train_paths)
    dino_test  = extract_dino_features(test_paths)
    
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.save(CACHE_DIR/'dino_train.npy', dino_train)
    np.save(CACHE_DIR/'dino_test.npy', dino_test)
    print("✅ DINOv2 features cached successfully.")
else:
    print("Loading DINOv2 features from cache...")
    dino_train = np.load(CACHE_DIR/'dino_train.npy')
    dino_test  = np.load(CACHE_DIR/'dino_test.npy')

print(f"DINOv2 Shapes -> Train: {dino_train.shape} | Test: {dino_test.shape}")


Loading DINOv2 features from cache...
DINOv2 Shapes -> Train: (4800, 768) | Test: (2058, 768)


In [12]:
# ============================================================
# CELL 7: FEATURE EXTRACTION — EfficientNet-B0 (1280-dim)
# ============================================================
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm.auto import tqdm
import numpy as np

def extract_cnn_features(image_paths, batch_size=128): # 3090 handles 128 easily at 224x224
    """Extract L2-normalized EfficientNet-B0 features (Global Average Pooled)."""
    print(f"Loading EfficientNet-B0 to {DEVICE}...")
    
    # Load weights using the modern Torchvision API
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model = efficientnet_b0(weights=weights)
    
    # Remove the final classification head to get the 1280-dim feature vector
    model.classifier = nn.Identity()
    model = model.to(DEVICE).eval()

    # Standard ImageNet preprocessing
    tfm = T.Compose([
        T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CNN Extraction'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            try:
                img = load_image_pil(p)
                if img is not None:
                    # Ensure 3-channel RGB for consistency
                    batch_imgs.append(tfm(img.convert('RGB')))
                else:
                    batch_imgs.append(torch.zeros(3, 224, 224))
            except Exception:
                batch_imgs.append(torch.zeros(3, 224, 224))
        
        batch = torch.stack(batch_imgs).to(DEVICE)
        
        # Use AMP for speed on RTX 3090
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                feats = model(batch)
            
        # L2 Normalize
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-6)
        feats_all.append(feats.float().cpu().numpy())

    # Strict VRAM management
    del model
    torch.cuda.empty_cache()
    return np.vstack(feats_all).astype(np.float32)

# ── Execution ───────────────────────────────────────────────
# Note: Using CACHE_DIR to match your DINOv2 cell
if FORCE_FRESH or not (CACHE_DIR/'cnn_train.npy').exists():
    print(f"Starting CNN extraction on {torch.cuda.get_device_name(0)}...")
    cnn_train = extract_cnn_features(train_paths)
    cnn_test  = extract_cnn_features(test_paths)
    
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.save(CACHE_DIR/'cnn_train.npy', cnn_train)
    np.save(CACHE_DIR/'cnn_test.npy', cnn_test)
    print("✅ CNN features cached successfully.")
else:
    print("Loading CNN features from cache...")
    cnn_train = np.load(CACHE_DIR/'cnn_train.npy')
    cnn_test  = np.load(CACHE_DIR/'cnn_test.npy')

print(f"CNN Shapes -> Train: {cnn_train.shape} | Test: {cnn_test.shape}")


Starting CNN extraction on NVIDIA GeForce RTX 3090...
Loading EfficientNet-B0 to cuda...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/jovyan/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 114MB/s]


CNN Extraction:   0%|          | 0/38 [00:00<?, ?it/s]

Loading EfficientNet-B0 to cuda...


CNN Extraction:   0%|          | 0/17 [00:00<?, ?it/s]

✅ CNN features cached successfully.
CNN Shapes -> Train: (4800, 1280) | Test: (2058, 1280)


In [17]:
# ============================================================
# CELL 8: FORENSIC FEATURE EXTRACTION (~114 dims)
# ============================================================
import io
import numpy as np
import pywt
from PIL import Image
from scipy.fftpack import fft2, fftshift
from tqdm.auto import tqdm
from scipy.ndimage import median_filter

# Target dimension based on your sub-functions (24 + 50 + 40 = 114)
FORENSIC_DIM = 114 

def load_image_np(path, size=(256, 256)):
    """Utility for forensic functions: loads image as normalized numpy array."""
    try:
        img = Image.open(str(path)).convert('RGB')
        if size:
            img = img.resize(size, Image.Resampling.LANCZOS)
        return np.array(img, dtype=np.float32)
    except Exception:
        return None

# ═══ 1. ELA Features (24 dims) ══════════════════════════════
def extract_ela_features(img_np):
    features = []
    img_pil = Image.fromarray(img_np.astype(np.uint8))
    for quality in [90, 75, 50]:
        buffer = io.BytesIO()
        img_pil.save(buffer, 'JPEG', quality=quality)
        buffer.seek(0)
        recompressed = np.array(Image.open(buffer), dtype=np.float64)
        ela = np.abs(img_np.astype(np.float64) - recompressed)
        for c in range(3):
            ch = ela[:, :, c]
            features.extend([np.mean(ch), np.std(ch)])
        ela_gray = np.mean(ela, axis=2)
        features.extend([np.percentile(ela_gray, 95), np.percentile(ela_gray, 5)])
    return np.array(features, dtype=np.float32)

# ═══ 2. FFT Features (50 dims) ══════════════════════════════
def extract_fft_features(img_np):
    gray = np.mean(img_np, axis=2)
    f_transform = fft2(gray)
    f_shift = fftshift(f_transform)
    magnitude = np.log1p(np.abs(f_shift))
    h, w = magnitude.shape
    cy, cx = h // 2, w // 2
    max_radius = min(cy, cx)
    n_bins = 30
    radial_profile = np.zeros(n_bins)
    for i in range(n_bins):
        r_inner = int(i * max_radius / n_bins)
        r_outer = int((i + 1) * max_radius / n_bins)
        y, x = np.ogrid[-cy:h-cy, -cx:w-cx]
        mask = (x*x + y*y >= r_inner**2) & (x*x + y*y < r_outer**2)
        if mask.any(): radial_profile[i] = np.mean(magnitude[mask])
    features = list(radial_profile)
    features.extend([np.mean(magnitude), np.std(magnitude), 
                     np.sum(magnitude[cy-10:cy+10, cx-10:cx+10]),
                     np.sum(magnitude) - np.sum(magnitude[cy-10:cy+10, cx-10:cx+10])])
    y_grid, x_grid = np.ogrid[-cy:h-cy, -cx:w-cx]
    r_sq = y_grid**2 + x_grid**2
    low_e = np.sum(magnitude[r_sq < (max_radius * 0.2)**2])
    mid_e = np.sum(magnitude[(r_sq >= (max_radius * 0.2)**2) & (r_sq < (max_radius * 0.5)**2)])
    high_e = np.sum(magnitude[r_sq >= (max_radius * 0.5)**2])
    total_e = low_e + mid_e + high_e + 1e-10
    features.extend([low_e/total_e, mid_e/total_e, high_e/total_e, high_e/(low_e+1e-10)])
    valid = radial_profile > 0
    slope = np.polyfit(np.log(np.arange(1, n_bins + 1)[valid]), np.log(radial_profile[valid]), 1)[0] if valid.sum() > 5 else 0.0
    features.append(slope)
    phase = np.angle(f_shift)
    features.extend([np.mean(phase), np.std(phase), np.mean(np.abs(np.diff(phase, axis=0))), np.mean(np.abs(np.diff(phase, axis=1)))])
    return np.array(features[:50], dtype=np.float32).flatten()

# ═══ 3. Noise Pattern Features (40 dims) ════════════════════
def extract_noise_features(img_np):
    gray = np.mean(img_np, axis=2)
    features = []
    for wavelet in ['db1', 'db2']:
        coeffs = pywt.dwt2(gray, wavelet)
        cA, (cH, cV, cD) = coeffs
        for detail in [cH, cV, cD]:
            features.extend([np.mean(np.abs(detail)), np.std(detail), np.percentile(np.abs(detail), 99), np.mean(detail**2)])
    denoised = median_filter(gray, size=3)
    noise = gray - denoised
    features.extend([np.mean(noise), np.std(noise), np.mean(noise**2), np.percentile(noise, 1), np.percentile(noise, 99)])
    return np.array(features[:40], dtype=np.float32)

def extract_forensic_features_single(path):
    img_np = load_image_np(path)
    if img_np is None: return np.zeros(FORENSIC_DIM, dtype=np.float32)
    combined = np.concatenate([extract_ela_features(img_np), extract_fft_features(img_np), extract_noise_features(img_np)])
    if len(combined) > FORENSIC_DIM: return combined[:FORENSIC_DIM]
    return np.pad(combined, (0, max(0, FORENSIC_DIM - len(combined))))

def extract_forensic_features_batch(paths):
    return np.vstack([extract_forensic_features_single(p) for p in tqdm(paths, desc='Forensics')])

# ── Execution & Cache ───────────────────────────────────────
if FORCE_FRESH or not (CACHE_DIR/'forensic_train.npy').exists():
    print("Extracting Forensic features...")
    forensic_train = extract_forensic_features_batch(train_paths)
    forensic_test  = extract_forensic_features_batch(test_paths)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.save(CACHE_DIR/'forensic_train.npy', forensic_train)
    np.save(CACHE_DIR/'forensic_test.npy', forensic_test)
else:
    print("Loading Forensic features from cache...")
    forensic_train = np.load(CACHE_DIR/'forensic_train.npy')
    forensic_test  = np.load(CACHE_DIR/'forensic_test.npy')

print(f"Forensic Shapes -> Train: {forensic_train.shape} | Test: {forensic_test.shape}")


Loading Forensic features from cache...
Forensic Shapes -> Train: (4800, 120) | Test: (2058, 120)


In [18]:
# ============================================================
# CELL 9: SHARED CV INFRASTRUCTURE (Renku/RTX 3090 Optimized)
# ============================================================

results_tracker = {}
oof_store       = {}
test_pred_store = {}

# ── Classical model CV ───────────────────────────────────────
def evaluate_cv(name, X, y, model_factory, n_splits=N_FOLDS, store_oof=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    tr_f1s, val_f1s, aucs = [], [], []

    print(f"\n{'='*62}\n  {name}\n{'='*62}")
    print(f"{'Fold':<5} {'Tr-F1':<8} {'Va-F1':<8} {'Gap':<7} {'AUC':<8} Status")
    print('-'*48)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        Xtr, Xv = X[tr_idx], X[val_idx]
        ytr, yv = y[tr_idx], y[val_idx]
        
        m = model_factory()
        m.fit(Xtr, ytr)
        
        tp = m.predict_proba(Xtr)[:, 1]
        vp = m.predict_proba(Xv)[:, 1]
        
        tf1 = f1_score(ytr, (tp >= 0.5).astype(int))
        vf1 = f1_score(yv,  (vp >= 0.5).astype(int))
        au  = roc_auc_score(yv, vp)
        gap = tf1 - vf1
        
        oof[val_idx] = vp
        tr_f1s.append(tf1); val_f1s.append(vf1); aucs.append(au)
        
        st = 'PASS' if gap < 0.08 else ('WARN' if gap < 0.10 else 'FAIL')
        print(f"{fold+1:<5} {tf1:<8.4f} {vf1:<8.4f} {gap:<7.4f} {au:<8.4f} {st}")

    mv = np.mean(val_f1s); sv = np.std(val_f1s)
    mt = np.mean(tr_f1s);  ma = np.mean(aucs)
    mg = mt - mv
    lb = 'PASS' if mg < 0.08 else ('WARN' if mg < 0.10 else 'FAIL')
    
    print('-'*48)
    print(f"MEAN  {mt:<8.4f} {mv:<8.4f} {mg:<7.4f} {ma:<8.4f} [{lb}]")
    print(f"STD            {sv:<8.4f}")

    result = {
        'name': name,
        'val_f1_mean': mv, 'val_f1_std': sv,
        'train_f1_mean': mt, 'gap': mg,
        'val_auc_mean': ma,
        'oof_proba': oof.copy() if store_oof else None
    }
    return result


# ── Fine-tune epoch runner ────────────────────────────────────
def run_epoch(model, loader, optimizer, scaler, criterion, is_train, use_mixup=False):
    model.train() if is_train else model.eval()
    tot_loss = 0.0; preds = []; trues = []
    
    # Use standard torch context managers
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)

            # Optimization: Using RTX 3090 Autocast for speed
            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                if is_train and use_mixup and random.random() < 0.5:
                    # Note: mixup_data must be defined in your augmentation cell
                    imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.3)
                    logits = model(imgs)
                    loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
                else:
                    logits = model(imgs)
                    loss = criterion(logits, labels.float() if isinstance(criterion, nn.BCEWithLogitsLoss) else labels)

            if is_train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            tot_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds.extend(probs.tolist())
            trues.extend(labels.cpu().numpy().tolist())

    f1_val = f1_score(trues, (np.array(preds) >= 0.5).astype(int), zero_division=0)
    return tot_loss / len(trues), f1_val, np.array(preds)


# ── Two-stage fine-tune for one fold ─────────────────────────
def train_finetune_fold(model, tr_ds, val_ds, criterion,
                        freeze_fn, unfreeze_fn, get_opt_fn,
                        fold_num, batch_size=32):
    import copy
    
    # Using 4 workers for Renku's CPU to feed the RTX 3090 faster
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=4, pin_memory=True, drop_last=True)
    va_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                       num_workers=4, pin_memory=True)

    # ── Stage 1: Head only ───────────────────────────────────
    freeze_fn(model)
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=0.01)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=10, eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

    best_f1 = 0; best_state = None; best_vp = None
    patience = 5; no_improve = 0

    for ep in range(1, 11):
        tl, tf, _ = run_epoch(model, tr_ld, opt1, scaler, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler, criterion, False)
        sch1.step()
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    s1_f1 = best_f1
    model.load_state_dict(best_state)

    # ── Stage 2: Unfreeze last blocks ────────────────────────
    unfreeze_fn(model)
    opt2 = get_opt_fn(model)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15, eta_min=1e-7)
    no_improve = 0

    for ep in range(1, 16):
        tl, tf, _ = run_epoch(model, tr_ld, opt2, scaler, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler, criterion, False)
        sch2.step()
        
        gap = tf - vf
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            best_vp = vp.copy()
        else:
            no_improve += 1
            
        if no_improve >= patience:
            break
        if gap > 0.12:
            print(f"   Fold {fold_num}: overfitting gap {gap:.4f} > 0.12, stopping Stage 2 Early")
            break

    model.load_state_dict(best_state)
    print(f"   Fold {fold_num}: Stage1 F1={s1_f1:.4f} -> Stage2 F1={best_f1:.4f}")
    return best_f1, best_vp, best_state

print("CV infrastructure ready for RTX 3090.")

CV infrastructure ready for RTX 3090.


In [22]:
# ── Define labels for the baseline models ────────────────────
# Ensure df_train was loaded in your earlier path-setup cell
y_all = df_train['ground_truth'].values

# Also ensure FORENSIC_DIM is defined for future forensic cells
FORENSIC_DIM = 114

In [23]:
# ============================================================
# CELL 10: CLASSICAL BASELINES (for comparison)
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

set_seeds()

# A. LogReg on CLIP
res_lr = evaluate_cv('LR-CLIP', clip_train, y_all,
    lambda: Pipeline([
        ('lr', LogisticRegression(C=1.0, penalty='l2', max_iter=1000,
                                  class_weight='balanced', solver='lbfgs',
                                  random_state=SEED))
    ]))
results_tracker['logreg_clip'] = res_lr
oof_store['logreg_clip'] = res_lr['oof_proba']

# B. SVM on CLIP
res_svm = evaluate_cv('SVM-CLIP', clip_train, y_all,
    lambda: Pipeline([
        ('sc', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, class_weight='balanced',
                    random_state=SEED))
    ]))
results_tracker['svm_clip'] = res_svm
oof_store['svm_clip'] = res_svm['oof_proba']

# C. SVM on DINOv2
res_dino = evaluate_cv('SVM-DINOv2', dino_train, y_all,
    lambda: Pipeline([
        ('sc', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, class_weight='balanced',
                    random_state=SEED))
    ]))
results_tracker['svm_dino'] = res_dino
oof_store['svm_dino'] = res_dino['oof_proba']

# D. SVM on CLIP + DINOv2 fused
# PCA is used here to manage the high dimensionality of the combined features
X_fused_cd = np.hstack([clip_train, dino_train])
res_fused = evaluate_cv('SVM-CLIP+DINOv2', X_fused_cd, y_all,
    lambda: Pipeline([
        ('sc', StandardScaler()),
        ('pca', PCA(n_components=256, random_state=SEED)),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                    probability=True, class_weight='balanced',
                    random_state=SEED))
    ]))
results_tracker['svm_clip_dino'] = res_fused
oof_store['svm_clip_dino'] = res_fused['oof_proba']

print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
baseline_keys = ['logreg_clip', 'svm_clip', 'svm_dino', 'svm_clip_dino']
for k in baseline_keys:
    if k in results_tracker:
        r = results_tracker[k]
        status = "PASS" if r['gap'] < 0.08 else "WARN"
        print(f"  {r['name']:<25} Val F1={r['val_f1_mean']:.4f}  Gap={r['gap']:.4f} [{status}]")


  LR-CLIP
Fold  Tr-F1    Va-F1    Gap     AUC      Status
------------------------------------------------
1     0.8558   0.8185   0.0373  0.9125   PASS
2     0.8510   0.8404   0.0106  0.9248   PASS
3     0.8492   0.8314   0.0178  0.9192   PASS
4     0.8549   0.8305   0.0244  0.9154   PASS
5     0.8549   0.8298   0.0252  0.9116   PASS
------------------------------------------------
MEAN  0.8532   0.8301   0.0231  0.9167   [PASS]
STD            0.0070  

  SVM-CLIP
Fold  Tr-F1    Va-F1    Gap     AUC      Status
------------------------------------------------
1     0.9769   0.8593   0.1176  0.9404   FAIL
2     0.9762   0.8654   0.1107  0.9455   FAIL
3     0.9754   0.8596   0.1157  0.9446   FAIL
4     0.9775   0.8547   0.1228  0.9360   FAIL
5     0.9735   0.8630   0.1105  0.9428   FAIL
------------------------------------------------
MEAN  0.9759   0.8604   0.1155  0.9419   [FAIL]
STD            0.0037  

  SVM-DINOv2
Fold  Tr-F1    Va-F1    Gap     AUC      Status
-------------------

In [40]:
# ============================================================
# CELL 11: CLIP ViT-L/14 FINE-TUNE — 5-Fold CV (Final Fix)
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import clip
import numpy as np
from tqdm.auto import tqdm

set_seeds()

N_TTA = 5  
N_FOLDS = 5 # Ensure this matches your global config

class CLIPFineTuner(nn.Module):
    def __init__(self, clip_visual, embed_dim=768):
        super().__init__()
        self.visual = clip_visual
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        f = self.visual(x).float()
        return self.head(f).squeeze(-1)

def freeze_clip(model):
    for p in model.visual.parameters():
        p.requires_grad = False

def unfreeze_clip_blocks(model, n=2):
    blocks = model.visual.transformer.resblocks
    for p in blocks[-n:].parameters():
        p.requires_grad = True
    if hasattr(model.visual, 'ln_post'):
        for p in model.visual.ln_post.parameters():
            p.requires_grad = True

def get_clip_optimizer(model, backbone_lr=5e-6, head_lr=1e-4):
    backbone_params = [p for n, p in model.visual.named_parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': 0.05},
        {'params': head_params,     'lr': head_lr,     'weight_decay': 0.01}
    ])

# FIX: Unpack tuple for Pydantic validation if your helper functions expect it
# If your helpers use A.Resize(size=size), they need to be updated to A.Resize(height=h, width=w)
clip_train_tfm = get_train_transform_albu(CLIP_MEAN, CLIP_STD, size=224)
clip_val_tfm   = get_val_transform_albu(CLIP_MEAN, CLIP_STD, size=224)
clip_tta_tfm   = get_tta_transform_albu(CLIP_MEAN, CLIP_STD, size=224)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
clip_oof = np.zeros(len(y_all))
clip_fold_f1s = []
clip_fold_models = []

print("="*60)
print(f"CLIP ViT-L/14 FINE-TUNE on {torch.cuda.get_device_name(0)}")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_paths, y_all)):
    print(f"\n-- Fold {fold+1}/{N_FOLDS} --")

    tr_paths_f   = [train_paths[i] for i in tr_idx]
    val_paths_f  = [train_paths[i] for i in val_idx]
    tr_labels_f  = y_all[tr_idx].tolist()
    val_labels_f = y_all[val_idx].tolist()

    tr_ds  = AlbuDataset(tr_paths_f, tr_labels_f, clip_train_tfm)
    val_ds = AlbuDataset(val_paths_f, val_labels_f, clip_val_tfm)

    _cm, _ = clip.load('ViT-L/14', device='cpu')
    model = CLIPFineTuner(_cm.visual).to(DEVICE)
    del _cm
    torch.cuda.empty_cache()

    n_pos = sum(tr_labels_f)
    n_neg = len(tr_labels_f) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    fold_f1, fold_vp, fold_state = train_finetune_fold(
        model, tr_ds, val_ds, criterion,
        freeze_fn=freeze_clip,
        unfreeze_fn=lambda m: unfreeze_clip_blocks(m, n=2),
        get_opt_fn=get_clip_optimizer,
        fold_num=fold+1, batch_size=32
    )

    clip_oof[val_idx] = fold_vp
    clip_fold_f1s.append(fold_f1)
    clip_fold_models.append(fold_state)

    del model
    torch.cuda.empty_cache()

# ── Test inference: 5 fold models × 5 TTA ────────────────────
print(f"\n-- Test Inference ({N_FOLDS} models x {N_TTA} TTA) --")
clip_test_proba = np.zeros(len(test_paths))

for fold, fold_state in enumerate(clip_fold_models):
    _cm, _ = clip.load('ViT-L/14', device='cpu')
    model = CLIPFineTuner(_cm.visual).to(DEVICE)
    model.load_state_dict(fold_state)
    model.eval()
    del _cm
    
    test_ds = AlbuDataset(test_paths, [-1]*len(test_paths), clip_tta_tfm)
    test_ld = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    fold_tta_preds = np.zeros(len(test_paths))
    for t in range(N_TTA):
        preds = []
        with torch.no_grad():
            for imgs, _ in test_ld:
                with torch.cuda.amp.autocast():
                    logits = model(imgs.to(DEVICE))
                preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        fold_tta_preds += np.array(preds)
    
    clip_test_proba += (fold_tta_preds / N_TTA)
    del model
    torch.cuda.empty_cache()

clip_test_proba /= N_FOLDS
clip_val_f1_mean = np.mean(clip_fold_f1s)
clip_oof_auc = roc_auc_score(y_all, clip_oof)

# Store results
results_tracker['clip_finetune'] = {
    'name': 'CLIP-FT (5-fold)',
    'val_f1_mean': clip_val_f1_mean,
    'val_auc_mean': clip_oof_auc,
    'oof_proba': clip_oof.copy()
}

print(f"\n✅ CLIP Fine-Tune Finished. OOF AUC: {clip_oof_auc:.4f}")


CLIP ViT-L/14 FINE-TUNE on NVIDIA GeForce RTX 3090

-- Fold 1/5 --
   Fold 1: Stage1 F1=0.8508 -> Stage2 F1=0.8792

-- Fold 2/5 --
   Fold 2: Stage1 F1=0.8651 -> Stage2 F1=0.8914

-- Fold 3/5 --
   Fold 3: Stage1 F1=0.8640 -> Stage2 F1=0.8939

-- Fold 4/5 --
   Fold 4: Stage1 F1=0.8614 -> Stage2 F1=0.8902

-- Fold 5/5 --
   Fold 5: Stage1 F1=0.8687 -> Stage2 F1=0.8973

-- Test Inference (5 models x 5 TTA) --

✅ CLIP Fine-Tune Finished. OOF AUC: 0.9630


In [44]:
# ============================================================
# CELL 12: DINOv2 ViT-B/14 FINE-TUNE — 5-Fold CV
# ============================================================
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

set_seeds()

class DINOv2FineTuner(nn.Module):
    def __init__(self, backbone, embed_dim=768):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        return self.head(f).squeeze(-1)

def freeze_dino(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

def unfreeze_dino_blocks(model, n=3):
    blocks = model.backbone.blocks
    for p in blocks[-n:].parameters():
        p.requires_grad = True
    if hasattr(model.backbone, 'norm'):
        for p in model.backbone.norm.parameters():
            p.requires_grad = True

def get_dino_optimizer(model, backbone_lr=5e-6, head_lr=1e-4):
    backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': 0.05},
        {'params': head_params,     'lr': head_lr,     'weight_decay': 0.01}
    ])

# ── DINOv2-specific transforms (518x518) — self-contained ────
dino_train_tfm = A.Compose([
    A.RandomResizedCrop(size=(518, 518), scale=(0.8, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussNoise(var_limit=(5, 30), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
    A.Normalize(mean=DINO_MEAN, std=DINO_STD),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
])
dino_val_tfm = A.Compose([
    A.Resize(height=518, width=518),
    A.Normalize(mean=DINO_MEAN, std=DINO_STD),
])
dino_tta_tfm = A.Compose([
    A.RandomResizedCrop(size=(518, 518), scale=(0.9, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=DINO_MEAN, std=DINO_STD),
])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
dino_oof = np.zeros(len(y_all))
dino_fold_f1s = []
dino_fold_models = []

print("=" * 60)
print(f"DINOv2 ViT-B/14 FINE-TUNE on {torch.cuda.get_device_name(0)}")
print("=" * 60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_paths, y_all)):
    print(f"\n-- Fold {fold+1}/{N_FOLDS} --")

    tr_paths_f   = [train_paths[i] for i in tr_idx]
    val_paths_f  = [train_paths[i] for i in val_idx]
    tr_labels_f  = y_all[tr_idx].tolist()
    val_labels_f = y_all[val_idx].tolist()

    tr_ds  = AlbuDataset(tr_paths_f, tr_labels_f, dino_train_tfm)
    val_ds = AlbuDataset(val_paths_f, val_labels_f, dino_val_tfm)

    backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                                  pretrained=True, num_classes=0)
    model = DINOv2FineTuner(backbone).to(DEVICE)

    n_pos = sum(tr_labels_f); n_neg = len(tr_labels_f) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    fold_f1, fold_vp, fold_state = train_finetune_fold(
        model, tr_ds, val_ds, criterion,
        freeze_fn=freeze_dino,
        unfreeze_fn=lambda m: unfreeze_dino_blocks(m, n=3),
        get_opt_fn=get_dino_optimizer,
        fold_num=fold+1, batch_size=16
    )

    dino_oof[val_idx] = fold_vp
    dino_fold_f1s.append(fold_f1)
    dino_fold_models.append(fold_state)

    del model, backbone
    torch.cuda.empty_cache()

# ── Test inference ───────────────────────────────────────────
print(f"\n-- Test Inference (5 models x {N_TTA} TTA) --")
dino_test_proba = np.zeros(len(test_paths))

for fold, fold_state in enumerate(dino_fold_models):
    print(f"   Inference Fold {fold+1}...")
    backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                                  pretrained=True, num_classes=0)
    model = DINOv2FineTuner(backbone).to(DEVICE)
    model.load_state_dict(fold_state)
    model.eval()

    test_ds = AlbuDataset(test_paths, [-1]*len(test_paths), dino_tta_tfm)
    test_ld = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

    fold_tta_preds = np.zeros(len(test_paths))
    for t in range(N_TTA):
        preds = []
        with torch.no_grad():
            for imgs, _ in test_ld:
                with torch.cuda.amp.autocast():
                    logits = model(imgs.to(DEVICE))
                preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        fold_tta_preds += np.array(preds)

    dino_test_proba += (fold_tta_preds / N_TTA)
    del model, backbone
    torch.cuda.empty_cache()

dino_test_proba /= N_FOLDS
dino_val_f1_mean = np.mean(dino_fold_f1s)
dino_oof_auc = roc_auc_score(y_all, dino_oof)

results_tracker['dino_finetune'] = {
    'name': 'DINOv2-FT (5-fold)',
    'val_f1_mean': dino_val_f1_mean,
    'val_auc_mean': dino_oof_auc,
    'oof_proba': dino_oof.copy()
}
test_pred_store['dino_finetune'] = dino_test_proba.copy()

print(f"\n✅ DINOv2 Fine-Tune Finished. OOF AUC: {dino_oof_auc:.4f}")

DINOv2 ViT-B/14 FINE-TUNE on NVIDIA GeForce RTX 3090

-- Fold 1/5 --
   Fold 1: Stage1 F1=0.8036 -> Stage2 F1=0.9052

-- Fold 2/5 --
   Fold 2: Stage1 F1=0.8092 -> Stage2 F1=0.8952

-- Fold 3/5 --
   Fold 3: Stage1 F1=0.8137 -> Stage2 F1=0.9039

-- Fold 4/5 --
   Fold 4: Stage1 F1=0.8445 -> Stage2 F1=0.9182

-- Fold 5/5 --
   Fold 5: Stage1 F1=0.8076 -> Stage2 F1=0.9016

-- Test Inference (5 models x 5 TTA) --
   Inference Fold 1...
   Inference Fold 2...
   Inference Fold 3...
   Inference Fold 4...
   Inference Fold 5...

✅ DINOv2 Fine-Tune Finished. OOF AUC: 0.9625


In [45]:
# ============================================================
# CELL 13: FORENSIC FEATURES → XGBoost (5-Fold CV)
# ============================================================
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
set_seeds()

vt = VarianceThreshold(threshold=1e-10)
forensic_train_clean = vt.fit_transform(forensic_train)
forensic_test_clean  = vt.transform(forensic_test)
n_kept = forensic_train_clean.shape[1]
print("="*60)
print(f"Forensic features: {forensic_train.shape[1]} -> {n_kept} (removed {forensic_train.shape[1] - n_kept} dead)")
print("="*60)

XGB_PARAMS = dict(
    n_estimators=500, 
    max_depth=4, 
    learning_rate=0.05,
    subsample=0.7, 
    colsample_bytree=0.7, 
    min_child_weight=5,
    reg_alpha=0.1, 
    reg_lambda=1.0, 
    gamma=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED, 
    tree_method='hist', 
    device='cuda'
)

res_forensic = evaluate_cv('XGB-Forensic', forensic_train_clean, y_all,
    lambda: Pipeline([
        ('sc', RobustScaler()),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ]))
results_tracker['xgb_forensic'] = res_forensic
oof_store['xgb_forensic'] = res_forensic['oof_proba']

X_forensic_cnn      = np.hstack([forensic_train_clean, cnn_train])
X_forensic_cnn_test = np.hstack([forensic_test_clean, cnn_test])

res_fc = evaluate_cv('XGB-Forensic+CNN', X_forensic_cnn, y_all,
    lambda: Pipeline([
        ('sc', RobustScaler()),
        ('pca', PCA(n_components=128, random_state=SEED)),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ]))
results_tracker['xgb_forensic_cnn'] = res_fc
oof_store['xgb_forensic_cnn'] = res_fc['oof_proba']

if res_fc['val_f1_mean'] > res_forensic['val_f1_mean']:
    best_forensic_key = 'xgb_forensic_cnn'
    X_train_f = X_forensic_cnn
    X_test_f  = X_forensic_cnn_test
    final_pipe = Pipeline([
        ('sc', RobustScaler()),
        ('pca', PCA(n_components=128, random_state=SEED)),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ])
else:
    best_forensic_key = 'xgb_forensic'
    X_train_f = forensic_train_clean
    X_test_f  = forensic_test_clean
    final_pipe = Pipeline([
        ('sc', RobustScaler()),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ])

print(f"\nFinal training for {best_forensic_key}...")
final_pipe.fit(X_train_f, y_all)
test_pred_store[best_forensic_key] = final_pipe.predict_proba(X_test_f)[:, 1]

print(f"\n✅ Best forensic model: {best_forensic_key}")
print(f"   Val F1 (Mean): {results_tracker[best_forensic_key]['val_f1_mean']:.4f}")
print(f"   Val AUC: {results_tracker[best_forensic_key]['val_auc_mean']:.4f}")

Forensic features: 120 -> 99 (removed 21 dead)

  XGB-Forensic
Fold  Tr-F1    Va-F1    Gap     AUC      Status
------------------------------------------------
1     0.9402   0.6512   0.2890  0.7285   FAIL
2     0.9378   0.6709   0.2669  0.7466   FAIL
3     0.9432   0.6637   0.2795  0.7655   FAIL
4     0.9520   0.6193   0.3327  0.6936   FAIL
5     0.9417   0.6723   0.2695  0.7301   FAIL
------------------------------------------------
MEAN  0.9430   0.6555   0.2875  0.7328   [FAIL]
STD            0.0196  

  XGB-Forensic+CNN
Fold  Tr-F1    Va-F1    Gap     AUC      Status
------------------------------------------------
1     0.6767   0.6726   0.0041  0.6808   PASS
2     0.6772   0.6706   0.0066  0.6451   PASS
3     0.6740   0.6850   -0.0111 0.6671   PASS
4     0.6766   0.6751   0.0015  0.6895   PASS
5     0.6766   0.6760   0.0005  0.7019   PASS
------------------------------------------------
MEAN  0.6762   0.6759   0.0003  0.6769   [PASS]
STD            0.0050  

Final training for x

In [46]:
# ============================================================
# CELL 14: MULTI-MODEL ENSEMBLE + THRESHOLD TUNING
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

set_seeds()

# 1. Identify models for ensemble
# Using a threshold of 0.60 for the forensic model to ensure it adds value
ensemble_keys = ['clip_finetune', 'dino_finetune']
if best_forensic_key in results_tracker and results_tracker[best_forensic_key]['val_f1_mean'] > 0.60:
    ensemble_keys.append(best_forensic_key)

print("="*60)
print("ENSEMBLE COMPOSITION")
print("="*60)
for k in ensemble_keys:
    r = results_tracker[k]
    print(f"  {r['name']:<30} | Val F1: {r['val_f1_mean']:.4f}")

# ── Strategy A: Weighted Average (Squared F1 Weighting) ────────────────
print("\n-- Strategy A: Weighted Average --")
f1_sq = {k: results_tracker[k]['val_f1_mean']**2 for k in ensemble_keys}
total_weight = sum(f1_sq.values())
weights = {k: v/total_weight for k, v in f1_sq.items()}

print("Calculated Weights:")
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} w={w:.4f}")

oof_blend_A = np.zeros(len(y_all))
for k, w in weights.items():
    oof_blend_A += w * oof_store[k]

# Find best threshold for Strategy A
best_thr_A = 0.5; best_f1_A = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (oof_blend_A >= t).astype(int))
    if f > best_f1_A:
        best_f1_A = f; best_thr_A = round(t, 2)
print(f"Result A: OOF F1={best_f1_A:.4f} @ Thr={best_thr_A}")

# ── Strategy B: LogReg Meta-Learner (Stacking) ──────────────────────────
print("\n-- Strategy B: LogReg Meta-Learner --")
X_meta = np.column_stack([oof_store[k] for k in ensemble_keys])

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
meta_oof = np.zeros(len(y_all))

for fold, (tr_idx, val_idx) in enumerate(skf_meta.split(X_meta, y_all)):
    meta_lr = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_lr.fit(X_meta[tr_idx], y_all[tr_idx])
    meta_oof[val_idx] = meta_lr.predict_proba(X_meta[val_idx])[:, 1]

# Find best threshold for Strategy B
best_thr_B = 0.5; best_f1_B = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (meta_oof >= t).astype(int))
    if f > best_f1_B:
        best_f1_B = f; best_thr_B = round(t, 2)
print(f"Result B: OOF F1={best_f1_B:.4f} @ Thr={best_thr_B}")

# ── Select Best Strategy and Generate Final Test Predictions ───────────
print("\n" + "="*60)
if best_f1_B >= best_f1_A:
    print(f"WINNER: Meta-Learner ({best_f1_B:.4f})")
    BEST_THRESHOLD = best_thr_B
    ensemble_method = 'meta'
    
    # Train meta-learner on all OOF data
    meta_final = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_final.fit(X_meta, y_all)
    
    # Predict on test set
    X_meta_test = np.column_stack([test_pred_store[k] for k in ensemble_keys])
    test_pred_store['ensemble'] = meta_final.predict_proba(X_meta_test)[:, 1]
    best_ensemble_f1 = best_f1_B
    final_oof = meta_oof
else:
    print(f"WINNER: Weighted Average ({best_f1_A:.4f})")
    BEST_THRESHOLD = best_thr_A
    ensemble_method = 'weighted_avg'
    
    # Blend test predictions
    test_blend = np.zeros(len(test_paths))
    for k, w in weights.items():
        test_blend += w * test_pred_store[k]
    test_pred_store['ensemble'] = test_blend
    best_ensemble_f1 = best_f1_A
    final_oof = oof_blend_A

# Store ensemble results
results_tracker['ensemble'] = {
    'name': f'Ensemble ({ensemble_method})',
    'val_f1_mean': best_ensemble_f1,
    'val_auc_mean': roc_auc_score(y_all, final_oof),
    'oof_proba': final_oof.copy()
}
oof_store['ensemble'] = final_oof.copy()

print(f"\n✅ FINAL ENSEMBLE COMPLETE")
print(f"   Method:    {ensemble_method}")
print(f"   Best Thr:  {BEST_THRESHOLD}")
print(f"   OOF F1:    {best_ensemble_f1:.4f}")
print(f"   OOF AUC:   {results_tracker['ensemble']['val_auc_mean']:.4f}")
print("="*60)

ENSEMBLE COMPOSITION
  CLIP-FT (5-fold)               | Val F1: 0.8904
  DINOv2-FT (5-fold)             | Val F1: 0.9048
  XGB-Forensic+CNN               | Val F1: 0.6759

-- Strategy A: Weighted Average --
Calculated Weights:
  dino_finetune                  w=0.3958
  clip_finetune                  w=0.3833
  xgb_forensic_cnn               w=0.2209


KeyError: 'clip_finetune'

In [47]:
# ============================================================
# CELL 14: MULTI-MODEL ENSEMBLE + THRESHOLD TUNING (Final Fix)
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

set_seeds()

# --- PRE-FLIGHT CHECK: Ensure OOFs are in the store ---
# This fixes the KeyError by manually mapping the variables if they are missing
if 'clip_oof' in locals(): oof_store['clip_finetune'] = clip_oof.copy()
if 'dino_oof' in locals(): oof_store['dino_finetune'] = dino_oof.copy()
if 'clip_test_proba' in locals(): test_pred_store['clip_finetune'] = clip_test_proba.copy()
if 'dino_test_proba' in locals(): test_pred_store['dino_finetune'] = dino_test_proba.copy()

# 1. Identify models for ensemble
ensemble_keys = ['clip_finetune', 'dino_finetune']
if 'best_forensic_key' in locals() and results_tracker[best_forensic_key]['val_f1_mean'] > 0.60:
    ensemble_keys.append(best_forensic_key)

print("="*60)
print("ENSEMBLE COMPOSITION")
print("="*60)
for k in ensemble_keys:
    r = results_tracker[k]
    print(f"  {r['name']:<30} | Val F1: {r['val_f1_mean']:.4f}")

# ── Strategy A: Weighted Average ─────────────────────────────
print("\n-- Strategy A: Weighted Average --")
f1_sq = {k: results_tracker[k]['val_f1_mean']**2 for k in ensemble_keys}
total_w = sum(f1_sq.values())
weights = {k: v/total_w for k, v in f1_sq.items()}

print("Calculated Weights:")
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} w={w:.4f}")

oof_blend_A = np.zeros(len(y_all))
for k, w in weights.items():
    oof_blend_A += w * oof_store[k]

best_thr_A = 0.5; best_f1_A = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (oof_blend_A >= t).astype(int))
    if f > best_f1_A:
        best_f1_A = f; best_thr_A = round(t, 2)
print(f"Result A: OOF F1={best_f1_A:.4f} @ Thr={best_thr_A}")

# ── Strategy B: LogReg Meta-Learner (Stacking) ────────────────
print("\n-- Strategy B: LogReg Meta-Learner --")
X_meta = np.column_stack([oof_store[k] for k in ensemble_keys])

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
meta_oof = np.zeros(len(y_all))

for fold, (tr_idx, val_idx) in enumerate(skf_meta.split(X_meta, y_all)):
    meta_lr = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_lr.fit(X_meta[tr_idx], y_all[tr_idx])
    meta_oof[val_idx] = meta_lr.predict_proba(X_meta[val_idx])[:, 1]

best_thr_B = 0.5; best_f1_B = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (meta_oof >= t).astype(int))
    if f > best_f1_B:
        best_f1_B = f; best_thr_B = round(t, 2)
print(f"Result B: OOF F1={best_f1_B:.4f} @ Thr={best_thr_B}")

# ── Select Final Strategy ─────────────────────────────────────
print("\n" + "="*60)
if best_f1_B >= best_f1_A:
    print(f"WINNER: Meta-Learner ({best_f1_B:.4f})")
    BEST_THRESHOLD = best_thr_B
    ensemble_method = 'meta'
    
    meta_final = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_final.fit(X_meta, y_all)
    
    X_meta_test = np.column_stack([test_pred_store[k] for k in ensemble_keys])
    test_pred_store['ensemble'] = meta_final.predict_proba(X_meta_test)[:, 1]
    best_ensemble_f1 = best_f1_B
    final_oof = meta_oof
else:
    print(f"WINNER: Weighted Average ({best_f1_A:.4f})")
    BEST_THRESHOLD = best_thr_A
    ensemble_method = 'weighted_avg'
    
    test_blend = np.zeros(len(test_paths))
    for k, w in weights.items():
        test_blend += w * test_pred_store[k]
    test_pred_store['ensemble'] = test_blend
    best_ensemble_f1 = best_f1_A
    final_oof = oof_blend_A

results_tracker['ensemble'] = {
    'name': f'Ensemble ({ensemble_method})',
    'val_f1_mean': best_ensemble_f1,
    'val_auc_mean': roc_auc_score(y_all, final_oof),
    'oof_proba': final_oof.copy()
}
oof_store['ensemble'] = final_oof.copy()

print(f"\n✅ FINAL ENSEMBLE READY")
print(f"   Method: {ensemble_method} | Thr: {BEST_THRESHOLD} | F1: {best_ensemble_f1:.4f}")
print("="*60)

ENSEMBLE COMPOSITION
  CLIP-FT (5-fold)               | Val F1: 0.8904
  DINOv2-FT (5-fold)             | Val F1: 0.9048
  XGB-Forensic+CNN               | Val F1: 0.6759

-- Strategy A: Weighted Average --
Calculated Weights:
  dino_finetune                  w=0.3958
  clip_finetune                  w=0.3833
  xgb_forensic_cnn               w=0.2209
Result A: OOF F1=0.9248 @ Thr=0.59

-- Strategy B: LogReg Meta-Learner --
Result B: OOF F1=0.9247 @ Thr=0.53

WINNER: Weighted Average (0.9248)

✅ FINAL ENSEMBLE READY
   Method: weighted_avg | Thr: 0.59 | F1: 0.9248


In [48]:
# ============================================================
# CELL 15: ANALYSIS, ABLATION, VISUALIZATION (Final Report)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, f1_score
import numpy as np

set_seeds()

# ── A. Summary Table ─────────────────────────────────────────
print("="*85)
header = f"{'Model':<32} {'Tr-F1':<8} {'Va-F1':<8} {'Std':<8} {'Gap':<7} {'AUC':<8} Status"
print(header)
print("="*85)

# Define the reporting order
order = ['logreg_clip', 'svm_clip', 'svm_dino', 'svm_clip_dino',
         'xgb_forensic', 'xgb_forensic_cnn',
         'clip_finetune', 'dino_finetune', 'ensemble']

for k in order:
    if k not in results_tracker:
        continue
    r = results_tracker[k]
    
    # Calculate gap if not present (Training vs Validation)
    train_f1 = r.get('train_f1_mean', 0.0)
    val_f1 = r.get('val_f1_mean', 0.0)
    gap = r.get('gap', abs(train_f1 - val_f1))
    std = r.get('val_f1_std', 0.0)
    auc = r.get('val_auc_mean', 0.0)
    
    # Status check for Overfitting
    status = 'PASS' if gap < 0.05 else ('WARN' if gap < 0.10 else 'FAIL')
    
    print(f"{r['name']:<32} {train_f1:<8.4f} {val_f1:<8.4f}"
          f" {std:<8.4f} {gap:<7.4f} {auc:<8.4f} {status}")

# ── B. Ablation Summary ───────────────────────────────────────
print("\n" + "-"*30)
print("   ABLATION STUDY SUMMARY")
print("-"*30)
ablation_map = [
    ("1. CLIP Frozen (Baseline):    ", 'svm_clip'),
    ("2. DINOv2 Frozen:             ", 'svm_dino'),
    ("3. CLIP Fine-tuned (5-fold):  ", 'clip_finetune'),
    ("4. DINOv2 Fine-tuned (5-fold):", 'dino_finetune'),
    ("5. Forensic XGBoost:          ", 'xgb_forensic'),
    ("6. Forensic+CNN Hybrid:       ", 'xgb_forensic_cnn'),
    ("7. FINAL ENSEMBLE:            ", 'ensemble'),
]

for label, key in ablation_map:
    val = results_tracker.get(key, {}).get('val_f1_mean', 'N/A')
    if isinstance(val, float):
        print(f"  {label} {val:.4f}")
    else:
        print(f"  {label} {val}")

# ── C. Confusion Matrix + ROC ────────────────────────────────
# Use the ensemble if it exists, otherwise fall back to best available
best_model_key = 'ensemble' if 'ensemble' in oof_store else 'dino_finetune'
best_oof = oof_store.get(best_model_key)
curr_threshold = BEST_THRESHOLD if 'BEST_THRESHOLD' in locals() else 0.5

oof_preds = (best_oof >= curr_threshold).astype(int)
cm = confusion_matrix(y_all, oof_preds)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Real', 'AI'], yticklabels=['Real', 'AI'])
axes[0].set_title(f'Confusion Matrix\n({best_model_key.upper()}, thr={curr_threshold})')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# Plot ROC Curves for major models
roc_keys = [k for k in ['clip_finetune', 'dino_finetune', 'ensemble'] if k in oof_store]
for k in roc_keys:
    fpr, tpr, _ = roc_curve(y_all, oof_store[k])
    auc_val = roc_auc_score(y_all, oof_store[k])
    axes[1].plot(fpr, tpr, label=f"{k} (AUC={auc_val:.4f})", lw=2)

axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc='lower right')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')

plt.tight_layout()
plt.show()

# ── D. Model Correlation ─────────────────────────────────────
print("\n" + "-"*30)
print("   MODEL CORRELATION MATRIX")
print("-"*30)
# Check correlation between the OOFs of the models in the ensemble
corr_keys = [k for k in ['clip_finetune', 'dino_finetune', 'xgb_forensic_cnn'] if k in oof_store]
if len(corr_keys) > 1:
    corr_data = np.column_stack([oof_store[k] for k in corr_keys])
    corr_matrix = np.corrcoef(corr_data.T)
    
    # Print formatted header
    print(f"{'':>22}", end='')
    for k in corr_keys:
        print(f" {k[:12]:>12}", end='')
    print()
    
    # Print rows
    for i, k in enumerate(corr_keys):
        print(f"{k[:20]:>22}", end='')
        for j in range(len(corr_keys)):
            print(f" {corr_matrix[i,j]:>12.4f}", end='')
        print()
    print("\n*Note: Lower correlation between models indicates a more robust ensemble.")
else:
    print("Not enough models in oof_store to calculate correlation.")

Model                            Tr-F1    Va-F1    Std      Gap     AUC      Status
LR-CLIP                          0.8532   0.8301   0.0070   0.0231  0.9167   PASS
SVM-CLIP                         0.9759   0.8604   0.0037   0.1155  0.9419   FAIL
SVM-DINOv2                       0.9790   0.8305   0.0122   0.1485  0.9153   FAIL
SVM-CLIP+DINOv2                  0.9722   0.8670   0.0039   0.1052  0.9468   FAIL
XGB-Forensic                     0.9430   0.6555   0.0196   0.2875  0.7328   FAIL
XGB-Forensic+CNN                 0.6762   0.6759   0.0050   0.0003  0.6769   PASS
CLIP-FT (5-fold)                 0.0000   0.8904   0.0000   0.8904  0.9630   FAIL
DINOv2-FT (5-fold)               0.0000   0.9048   0.0000   0.9048  0.9625   FAIL
Ensemble (weighted_avg)          0.0000   0.9248   0.0000   0.9248  0.9761   FAIL

------------------------------
   ABLATION STUDY SUMMARY
------------------------------
  1. CLIP Frozen (Baseline):     0.8604
  2. DINOv2 Frozen:              0.8305
  3. CLIP

In [49]:
# ============================================================
# CELL 16: FINAL SUBMISSION (Submission-Ready)
# ============================================================
import pandas as pd
import numpy as np
import os

set_seeds()

# 1. Select the Absolute Best Performer
# We compare the Ensemble against the best single fine-tuned models
best_key = 'ensemble'
available_models = ['ensemble', 'clip_finetune', 'dino_finetune']
current_best_f1 = 0.0

for k in available_models:
    if k in results_tracker:
        model_f1 = results_tracker[k]['val_f1_mean']
        if model_f1 > current_best_f1:
            current_best_f1 = model_f1
            best_key = k

print("="*60)
print(f"FINAL MODEL SELECTION: {best_key}")
print(f"Validation F1 Score:   {results_tracker[best_key]['val_f1_mean']:.4f}")
print("="*60)

# 2. Extract Predictions and Apply Threshold
test_proba = test_pred_store[best_key]

# Use the tuned threshold if available, otherwise default to 0.5
thr = BEST_THRESHOLD if 'BEST_THRESHOLD' in locals() else 0.5
print(f"Applying Threshold:    {thr}")

preds_binary = (test_proba >= thr).astype(int)

# 3. Build Submission DataFrame
# Assumes df_test is already loaded and contains 'image_id'
submission = pd.DataFrame({
    'image_id':     df_test['image_id'].values,
    'ground_truth': preds_binary
})

# 4. Strict Sanity Checks
print("\nRunning Sanity Checks...")

# Check shape (Adjust 2058 if your specific test set size differs)
expected_rows = len(df_test)
assert submission.shape == (expected_rows, 2), f"Error: Shape {submission.shape} != ({expected_rows}, 2)"

# Check for NaNs
assert not submission.isnull().any().any(), "Error: Submission contains null values!"

# Check for data types
assert submission['ground_truth'].isin([0, 1]).all(), "Error: Predictions must be 0 or 1 only!"

# Check for duplicate IDs
assert submission['image_id'].is_unique, "Error: Duplicate image_ids found in submission!"

n0 = (submission['ground_truth'] == 0).sum()
n1 = (submission['ground_truth'] == 1).sum()

print(f"✅ Sanity checks PASSED")
print(f"   Total Samples: {len(submission)}")
print(f"   Real (0):      {n0} ({n0/len(submission):.1%})")
print(f"   AI   (1):      {n1} ({n1/len(submission):.1%})")

# 5. Save to CSV
out_path = 'submission.csv' # Standard Kaggle/Renku path
submission.to_csv(out_path, index=False)

print(f"\nSuccessfully saved to: {os.path.abspath(out_path)}")
print("\nFirst 5 rows of submission:")
print("-" * 30)
print(submission.head())
print("-" * 30)

print(f"\nFinal Configuration: {best_key} | Val F1={results_tracker[best_key]['val_f1_mean']:.4f} | Thr={thr}")

FINAL MODEL SELECTION: ensemble
Validation F1 Score:   0.9248
Applying Threshold:    0.59

Running Sanity Checks...
✅ Sanity checks PASSED
   Total Samples: 2058
   Real (0):      1087 (52.8%)
   AI   (1):      971 (47.2%)

Successfully saved to: /home/jovyan/work/data/submission.csv

First 5 rows of submission:
------------------------------
                                   image_id  ground_truth
0  3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg             1
1  2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg             0
2  01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg             0
3  ac784910-b461-498d-b3a8-50b1e4116b11.jpg             0
4  6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg             0
------------------------------

Final Configuration: ensemble | Val F1=0.9248 | Thr=0.59
